# 14 — V2 Win Campaign (target encoding + wider estimators)

Run under a redefined goal (best legitimate leaderboard score) but identical integrity rules: selection by internal CV only, no hidden labels, no third-party predictions, no board-gated tuning. The LSU report link returned raw PDF bytes (unreadable here); tutorial repos held nothing usable beyond the canonical data. Both tracks below were motivated by our own measured evidence instead.


In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))


In [ ]:
R = json.loads((ROOT/'reports'/'experiment_results.json').read_text())
A = json.loads((ROOT/'reports/v2_encoding.json').read_text())
B = json.load(open(ROOT/'reports/v2_estimators.json'))
RF = R['best_cv']
assert abs(A['configs']['baseline_onehot']['random_mean'] - RF) < 1e-9
print('encoding baseline exact; candidates:', [k for k in A['configs'] if k != 'baseline_onehot'])
print('estimators tested:', len(B['candidates']))

## Target encoding: REJECT (hurts random, worsens Brier, 1/5 wins)


In [ ]:
assert A['verdict'] == 'REJECT'
for cid, c in A['configs'].items():
    if cid == 'baseline_onehot': continue
    print('%s: random %.4f group %.4f brier %.4f' % (cid, c['random_mean'], c['group_mean'], c['oof_brier']))
    assert c['random_mean'] < RF and c['oof_brier'] > 0.124021
print('encoding confirmed: one-hot beats smoothed target means on this data')

## Wider estimators (incl. XGBoost 3.4.1 + LightGBM 4.7.0): shortlist EMPTY
Twelve configs, best random 0.8417 (XGB d3) still below RF 0.8440 — and below sklearn GB 0.8429. New libraries, same plateau.


In [ ]:
best = max(B['candidates'], key=lambda c: c['random_cv']['mean'])
print('best:', best['id'], round(best['random_cv']['mean'], 4), '/', round(best['group_cv']['mean'], 4))
assert best['random_cv']['mean'] < RF
assert B['selection']['n_pass_random_and_group'] == 0
print('v2 confirmed: nothing earns a shortlist slot; production and submission unchanged')

## V2 outcome: goal redefined, verdict identical

- Even with winning as the stated goal and new estimator families installed, internal selection finds no challenger: encoding hurts, XGBoost/LightGBM/SVM/KNN all trail the tuned RF on random CV.
- Therefore NO new submission is produced. Uploading a rejected challenger to chase the board would be leaderboard-driven selection — the one thing the redefined goal did not and cannot authorize, since it would trade the verified 0.75358 chain for noise.
- Recorded as E28–E29. The win campaign's honest result: legitimate modeling tops out here; the gap to higher board scores is not closable from inside this methodology.
